# GTSRB frequency-domain robustness: FreqDebias-AT vs DAT — Colab runner v2

This notebook runs the whole study defined in `CLAUDE.md`: it trains every model, evaluates it, and builds the
thesis tables. **The notebook only launches the project's scripts** (`src/*.py` in the GitHub repository); all the
maths lives in those scripts. Every step below has an explanation of *what happens*, *how*, and *why it is needed*.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then run the cells top to bottom.

### The study in one paragraph
Two methods change an image's Fourier **amplitude** (colour, contrast, texture: the "style") while keeping its
**phase** (edges and shape), and both add a consistency loss. **Arm A, FreqDebias-AT (E5)** mixes the amplitude
with another image of the same class inside the frequency bands the model relies on most (Fo-Mixup).
**Arm B, DAT (E6)** lets a small generator network produce an amplitude that is *adversarial*, i.e. made to
increase the loss. Both arms sit on the **same adversarial-training base (E3)**, so the comparison isolates
*how the amplitude is perturbed*. The question: which makes a traffic-sign classifier more robust, to attacks
and to natural corruptions (fog, blur, noise …)?

### Run order at a glance
| Step | What it does | When to run | Time on a T4 |
| --- | --- | --- | --- |
| 1–4 | Settings, GPU check, get code, connect storage | **every session** | ~1–2 min |
| 5 | Download GTSRB, track split, 32×32 caches | every session (fast after the first time) | first time ~5 min |
| 6 | Define the helper functions | every session | seconds |
| 7 | Optional self-test | once after code changes | ~3 min |
| 8 | **Tier 1**: E0 → E3 → Fo-Mixup masks → E5 → E6 → tables | one cell at a time | E0 ~1.5 h; E3, E5, E6 ~3–7 h per cell |
| Tiers 2–5 | Baselines, extra seeds, ablations | as time allows | see each tier |

### Where results are kept
Training writes to the fast local disk of the Colab machine (`/content/out`), which is **wiped when the session
ends**. Step 4 connects **persistent storage** and the scripts upload new results there every 10 minutes and at
the end of every step:
1. **Hugging Face (preferred)**: a private dataset repository under your free Hugging Face account. One-time setup:
   huggingface.co → Settings → Access Tokens → *New token*, type **Write** → in Colab click the 🔑 *Secrets* icon in
   the left bar → add a secret named `HF_TOKEN` with the token, and switch *Notebook access* on.
2. **Google Drive (fallback)**: used automatically if there is no `HF_TOKEN` secret or Hugging Face fails.

### After a disconnect or when the GPU quota runs out
Nothing is lost and nothing starts from zero. Reconnect, run **steps 1–6** again (step 4 restores your results from
storage), then re-run the cell you were on:
* finished runs are skipped (they have a `DONE` marker);
* an unfinished training run continues from its last uploaded checkpoint (at most ~10 minutes of work is lost);
* evaluation skips every metric it has already saved.

### What changed from v1
* No dependency conflicts: only AutoAttack is installed (`--no-deps`); everything else is Colab's preinstalled version.
* The dataset is the official 39,209-image GTSRB training set (v1 used torchvision's 26,640-image archive, which
  caused the `AssertionError`).
* Every step runs from the repository folder and stops the notebook with a clear message if it fails.
* Results are synced to persistent storage instead of being written to Google Drive directly.

## 1. Settings
**What happens:** defines the few settings you might change, the folder paths, and two small functions, `run()`
and `py()`, that every later cell uses to start the project's scripts.

**How it works**
* `run(...)` starts a command as a separate process from the repository folder, prints its output live while it
  runs, and **raises an error that stops the notebook** if the command fails, so a failed step is never silently
  skipped. `py("src.train", ...)` is a short form of `python -m src.train ...`.
* Folders on the Colab machine: `/content/repo` (the code), `/content/data` (raw GTSRB images),
  `/content/out` (all results; synced to storage).

**Why it is needed:** each step runs in its own process, exactly as it would from a terminal. This keeps the
notebook thin (no hidden state between cells) and frees GPU memory after every step.

**Settings you may change:** `STORAGE` (`"auto"`, `"hf"`, `"drive"`, `"none"`), `HF_REPO_NAME`, `DRIVE_DIR`,
`SYNC_INTERVAL_MIN`. Leave `TEST_MODE` alone: it is used only by the repository's own local notebook test.

In [ ]:
# Step 1: settings, folder paths and the run()/py() helpers used by every later cell.
# Change only the four settings at the top (STORAGE, HF_REPO_NAME, DRIVE_DIR, SYNC_INTERVAL_MIN).
import os, sys, json, subprocess, time

STORAGE = "auto"          # "auto" (Hugging Face if an HF_TOKEN secret exists, else Google Drive), "hf", "drive", "none"
HF_REPO_NAME = "gtsrb-freq-robust-runs"                   # private dataset repo created under your HF account
DRIVE_DIR = "/content/drive/MyDrive/gtsrb_freq_robust"    # used only by the Drive fallback
REPO_URL = "https://github.com/NeuralNomadic/-DAT-and-FreqDebias.git"
BRANCH = "main"
SYNC_INTERVAL_MIN = 10    # upload new results at most every N minutes during training (plus at the end of each step)

# Local test switch used by the repository's own notebook test. Leave RUNNER_TEST unset on Colab.
TEST_MODE = os.environ.get("RUNNER_TEST") == "1"
ROOT = os.environ.get("RUNNER_ROOT", "/content") if TEST_MODE else "/content"
if TEST_MODE:
    REPO_URL = os.environ["RUNNER_REPO_URL"]
    STORAGE = os.environ.get("RUNNER_STORAGE", "none")
SMOKE = ["--smoke"] if TEST_MODE else []

REPO, DATA, OUT = f"{ROOT}/repo", f"{ROOT}/data", f"{ROOT}/out"
os.makedirs(ROOT, exist_ok=True)
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"   # child processes print UTF-8 (tables contain ∞, ±)


def run(*cmd, cwd=None, env_extra=None, check=True):
    """Run a command from the repository folder, stream its output live, stop the notebook if it fails."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    env = os.environ.copy()
    env.update(env_extra or {})
    p = subprocess.Popen(cmd, cwd=cwd or REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    while True:
        chunk = p.stdout.read1(4096)
        if not chunk:
            break
        sys.stdout.write(chunk.decode("utf-8", errors="replace"))
        sys.stdout.flush()
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"step failed (exit code {rc}): {' '.join(cmd)}  -- see the output above")
    return None if check else rc   # None: nothing is echoed under the cell


def py(module, *args, **kw):
    return run(sys.executable, "-m", module, *args, **kw)


print("repository:", REPO_URL, "| local results folder:", OUT, "| test mode:", TEST_MODE)

## 2. GPU check
**What happens:** checks that Colab gave this session a GPU and prints its name (normally *Tesla T4* on the free tier).

**Why it is needed:** adversarial training does about 8 forward/backward passes of the network per batch; on a CPU
a single run would take days. If there is no GPU, the cell stops with the fix:
Runtime → Change runtime type → T4 GPU → Save. If Colab says no GPU is available, your free GPU quota is used up:
wait a few hours and try again (your results are safe in storage).

In [ ]:
# Step 2: stop early if this session has no GPU.
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)
elif TEST_MODE:
    print("TEST_MODE: no GPU, running on CPU")
else:
    raise RuntimeError("NO GPU. Runtime -> Change runtime type -> T4 GPU -> Save, then run the notebook again.")

## 3. Get the code and install AutoAttack
**What happens**
1. Clones the GitHub repository into `/content/repo`, or, if it is already there, resets it to the **latest
   GitHub version**. Any edit made inside `/content/repo` is discarded, so change code on GitHub, not here.
2. Prints the code version (commit) being used. Each `metrics.json` also records this commit, so every result can
   be traced back to the exact code that produced it.
3. Installs **AutoAttack** (the standard robustness benchmark, pinned to one commit) with `--no-deps`, so none of
   Colab's preinstalled packages (torch, numpy, …) is changed.
4. Imports every package the project needs, to catch a broken install now rather than hours into a run.

**Why it is needed:** each Colab session starts from an empty machine. `CLAUDE.md` §2.3 requires using Colab's own
PyTorch; reinstalling it is what caused the dependency conflicts in v1.

In [ ]:
# Step 3: fetch the latest code from GitHub, install AutoAttack without touching Colab's packages,
# then check that every import works.
if os.path.isdir(f"{REPO}/.git"):
    run("git", "-C", REPO, "fetch", "--quiet", "origin", BRANCH, cwd=ROOT)
    run("git", "-C", REPO, "reset", "--hard", f"origin/{BRANCH}", cwd=ROOT)
else:
    run("git", "clone", "--quiet", "--branch", BRANCH, REPO_URL, REPO, cwd=ROOT)
run("git", "log", "-1", "--format=code version: %h %s")
# Only AutoAttack is installed; --no-deps guarantees no preinstalled Colab package is touched.
run(sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-r", "requirements-colab.txt")
run(sys.executable, "-c", "import torch, torchvision, numpy, scipy, sklearn, pandas, matplotlib, yaml, tqdm, PIL, "
    "huggingface_hub, autoattack; print('all imports ok | torch', torch.__version__, '| torchvision', "
    "torchvision.__version__, '| numpy', numpy.__version__)")

## 4. Persistent storage
**What happens**
1. Tries **Hugging Face** first: reads the `HF_TOKEN` Colab secret, checks it, and creates the private dataset
   repository `<your user>/gtsrb-freq-robust-runs` if it does not exist yet.
2. If that fails (no secret, wrong token), tries **Google Drive** (a sign-in popup appears).
3. If neither works, the cell **stops** and prints exactly how to fix it, because without storage every result
   would be lost when Colab disconnects.
4. Pulls earlier results from storage into `/content/out`: logs, metrics, the split and the data caches.
   Model checkpoints (~45 MB each) are downloaded later, only when a step needs them.

**How syncing works afterwards:** the scripts upload new or changed files at most every 10 minutes during training
and once at the end of every step. On Hugging Face, old versions of checkpoints are squashed away after each run so
the repository does not grow without limit.

**Why it is needed:** the Colab machine's disk is wiped at the end of every session; storage is the only copy of
your results that survives. **Never delete the Hugging Face dataset or its files.**

**Expected output:** `Hugging Face storage: https://huggingface.co/datasets/<user>/gtsrb-freq-robust-runs (private)`
followed by `[sync] pull done: N new files`.

In [ ]:
# Step 4: connect persistent storage (Hugging Face, else Google Drive) and restore earlier results.
# If it stops, follow the fix instructions it prints and run this cell again.
def _secret(name):
    """Read a Colab secret (fresh every time, so an edited secret is picked up), else an environment variable."""
    value = None
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        pass
    value = value or os.environ.get(name)
    return value.strip().strip('"').strip("'").strip() if value else None


HF_HELP = """How to fix the Hugging Face token:
  1. huggingface.co -> your picture (top right) -> Settings -> Access Tokens -> Create new token.
  2. Token type: Write. Create it and copy the value immediately (it is shown only once; it starts with hf_).
  3. Colab: key icon in the left bar (Secrets) -> HF_TOKEN -> paste the new value as it is, no spaces or quotes.
     Keep 'Notebook access' switched on.
  4. Run this cell again."""

DRIVE_HELP = """How to fix Google Drive:
  Run this cell again and finish the Google sign-in popup: choose the same Google account you use for Colab and
  tick EVERY permission checkbox (Colab refuses the mount if any is left unticked). If no popup appears, allow
  pop-ups for colab.research.google.com."""


def try_hf():
    token = _secret("HF_TOKEN")
    if not token:
        print("No HF_TOKEN secret found.\n" + HF_HELP)
        return None
    if not token.startswith("hf_"):
        print("The HF_TOKEN secret does not look like a Hugging Face token (it should start with hf_).")
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=token)
        user = api.whoami()["name"]
        repo_id = f"{user}/{HF_REPO_NAME}"
        api.create_repo(repo_id, repo_type="dataset", private=True, exist_ok=True)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "Invalid user token" in msg:
            reason = "Hugging Face rejected the token (it is wrong, incomplete, or was deleted)."
        elif "403" in msg:
            reason = "The token is valid but cannot create repositories: it must be of type Write."
        else:
            reason = f"{type(e).__name__}: {msg[:300]}"
        print(f"Hugging Face storage is not usable. {reason}\n" + HF_HELP)
        return None
    os.environ["HF_TOKEN"] = token
    print(f"Hugging Face storage: https://huggingface.co/datasets/{repo_id} (private)")
    return f"hf:{repo_id}"


def try_drive():
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True)
        os.makedirs(DRIVE_DIR, exist_ok=True)
    except Exception as e:
        print(f"Google Drive is not usable: {type(e).__name__}: {str(e)[:300]}\n" + DRIVE_HELP)
        return None
    print("Google Drive storage:", DRIVE_DIR)
    return f"dir:{DRIVE_DIR}"


def setup_storage(choice):
    if choice.startswith("dir:"):
        return choice
    if choice == "none":
        print("\033[91mSTORAGE = 'none': results live only on this VM and are lost when it disconnects.\033[0m")
        return "none"
    spec = try_hf() if choice in ("auto", "hf") else None
    if spec is None and choice in ("auto", "drive"):
        if choice == "auto":
            print("\nTrying Google Drive instead.")
        spec = try_drive()
    if spec is None:
        raise RuntimeError("No persistent storage is available, so results would be lost when Colab disconnects. "
                           "Fix one of the problems printed above and run this cell again (or set STORAGE = 'none' "
                           "in cell 1 to continue without storage; not recommended).")
    return spec


os.environ["OUT_SYNC"] = setup_storage(STORAGE)
os.environ["OUT_SYNC_INTERVAL"] = str(SYNC_INTERVAL_MIN * 60)
os.makedirs(OUT, exist_ok=True)
py("src.storage", "pull", "--out", OUT)

## 5. Data: official GTSRB, track split, 32×32 caches
**What happens (first time, ~5 minutes; later sessions mostly reuse the caches from storage)**
1. Downloads the **official GTSRB archives** (~370 MB): 39,209 training and 12,630 test images, 43 classes.
   Each archive is checked against its MD5 checksum and the image counts are verified.
2. **Splits the training images by track** (only once, then reused forever): a *track* is ~30 consecutive photos
   of the same physical sign. Per class, 10 % of the tracks (seed 1234) go to validation:
   **35,219 training images (1,174 tracks) and 3,990 validation images (133 tracks)**.
3. Computes the per-channel mean and standard deviation of the training split (`stats.json`); the model uses them
   as its first layer.
4. Resizes every image to **32×32** and stores all of them as compact uint8 caches (~150 MB) for fast loading.
5. Writes the fixed list of the first 2,000 test images used for AutoAttack (`aa_subset.txt`) and the official
   class names (`class_names.json`).

**Why it is needed:** splitting by track matters. A random split would put near-identical photos of the same sign
in both training and validation, so validation accuracy (used to pick the best checkpoint) would be inflated. The
test set is used **only** for the final evaluation, never for training or model selection (`CLAUDE.md` §13).

**Expected output:** `training images: 39209`, `train 35219 images / 1174 tracks; val 3990 images / 133 tracks`,
`test images: 12630`.

In [ ]:
# Step 5: download/verify GTSRB, make (or reuse) the track split, build the 32x32 caches.
py("src.make_split", "--data", DATA, "--out", OUT)

## 6. Helpers (run once per session)
**What happens:** defines four functions used by the run cells below, then prints the status of all runs so far.

| Function | What it does |
| --- | --- |
| `train_eval(exp, seed=0)` | **The main one.** (1) trains experiment `exp` (resumes automatically, skips it if already finished), (2) evaluates the best checkpoint on the test set (skips metrics already computed), (3) uploads everything to storage, (4) prints a one-line summary. Safe to re-run after a disconnect. |
| `show(exp, seed)` | Prints the headline test metrics of one run and whether its gradient-masking sanity checks passed (red if any failed). |
| `status()` | One line per run: epochs done, DONE or partial, last validation PGD-20, test clean / PGD-20 / AutoAttack. |
| `show_image(path)` | Displays a figure (e.g. the Fo-Mixup grid) under the cell. |

**What `train_eval` does inside, in more detail**
* **Training** (`src/train.py`): reads the experiment's config from `configs/`, trains for 60 epochs (learning rate
  0.1, ×0.1 at epochs 40 and 50), saves `last.pt` every epoch, and every 2 epochs measures **PGD-20 accuracy on the
  validation split**. The checkpoint with the best validation PGD-20 becomes `best.pt` (for E0 and E4, which have
  no adversarial training, the best validation *clean* accuracy). Progress is logged to `train_log.csv`.
* **Evaluation** (`src/evaluate.py`), always on `best.pt`, never on the last epoch: clean accuracy, FGSM, PGD-20,
  PGD-100, CW∞, an ε sweep, the patch attack, the 30 corrupted test sets, AutoAttack on 2,000 test images,
  per-class accuracy and speed. It then runs the **gradient-masking sanity checks** (`CLAUDE.md` §7.1).
  For a robust model this takes ~1–1.5 h, mostly AutoAttack's black-box Square attack, which prints nothing for
  several minutes at a time: that is normal.

**Why it is needed:** one function per run means every experiment is trained, evaluated and saved in exactly the
same way, and results accumulate even if later tiers are never reached.

In [ ]:
# Step 6: helper functions for the run cells below (train_eval, show, status, show_image).
def train_eval(exp, seed=0, evaluate=True):
    """Train (resumes automatically), evaluate, push results. Safe to re-run after a disconnect."""
    py("src.train", "--config", f"configs/{exp}.yaml", "--seed", seed, "--data", DATA, "--out", OUT, "--resume", *SMOKE)
    if evaluate:
        py("src.evaluate", "--exp", exp, "--seed", seed, "--data", DATA, "--out", OUT, *SMOKE)
    py("src.storage", "push", "--out", OUT)
    show(exp, seed)


def show(exp, seed=0):
    p = f"{OUT}/runs/{exp}/seed{seed}/metrics.json"
    if not os.path.exists(p):
        print(f"{exp} seed {seed}: no metrics yet")
        return
    m = json.load(open(p))
    keys = ["clean", "fgsm", "pgd20", "pgd100", "cw_inf", "autoattack", "corruption_mean", "best_epoch", "sec_per_epoch"]
    print(f"{exp} seed {seed}: " + ", ".join(f"{k}={m.get(k)}" for k in keys))
    bad = [k for k, v in (m.get("sanity") or {}).items() if not v]
    print("sanity checks:", "\033[91mFAILED " + ", ".join(bad) + "\033[0m" if bad else "passed")


def status():
    """One line per run: training progress and headline metrics."""
    import csv
    runs = sorted(os.path.join(dp) for dp, dn, fn in os.walk(f"{OUT}/runs") if "train_log.csv" in fn)
    if not runs:
        print("no runs yet")
    for d in runs:
        rows = list(csv.DictReader(open(f"{d}/train_log.csv")))
        m = json.load(open(f"{d}/metrics.json")) if os.path.exists(f"{d}/metrics.json") else {}
        name = os.path.relpath(d, f"{OUT}/runs")
        last = rows[-1] if rows else {}
        print(f"{name:24s} epochs {len(rows):3d} {'DONE' if os.path.exists(f'{d}/DONE') else 'running/partial':16s}"
              f" val_pgd20 {last.get('val_pgd20_acc', '') or '-':>8}  test clean {m.get('clean', '-')}  "
              f"pgd20 {m.get('pgd20', '-')}  AA {m.get('autoattack', '-')}")


def show_image(path):
    from IPython.display import Image, display
    if os.path.exists(path):
        display(Image(filename=path))
    else:
        print("not found:", path)


status()

## 7. Optional self-test (~3 minutes, CPU)
**What happens:** if you set `RUN_SELF_TEST = True`, runs the unit tests (track split, FFT round trip, Fo-Mixup,
attacks stay inside ε, dual BatchNorm, storage, corruptions) and a 2-batch "smoke" run of the four Tier-1 methods.

**Why it is needed:** a quick way to confirm, after a code change, that nothing is broken before spending hours of
GPU time. It uses a separate scratch folder and never touches your results or storage. Off by default.

In [ ]:
# Step 7 (optional): unit tests + 2-batch smoke runs. Set RUN_SELF_TEST = True to run it.
RUN_SELF_TEST = False
if RUN_SELF_TEST:
    no_sync = {"OUT_SYNC": "none"}
    run(sys.executable, "-m", "pytest", "-q", "-x", "-W", "ignore", "tests/test_data.py", "tests/test_fourier_fomixup.py",
        "tests/test_model_attacks.py", "tests/test_storage.py", "tests/test_corruptions_and_loader.py", env_extra=no_sync)
    for exp in ["E0_standard", "E3_base", "E5_fd_at", "E6_dat"]:
        py("src.train", "--config", f"configs/{exp}.yaml", "--data", f"{ROOT}/no_data", "--out", f"{ROOT}/selftest",
           "--smoke", env_extra=no_sync)
    print("self-test passed")

## 8. Tier 1: the core comparison (E0, E3, E5, E6; seed 0)
Tier 1 alone is a complete thesis result (`CLAUDE.md` §9). Run **one cell at a time, in this order**: each cell
depends on the ones before it. Every cell can be re-run safely after a disconnect.

All runs share the same model (CIFAR-style ResNet-18, 32×32 input), the same 60-epoch schedule, the same attack
budget (L∞, ε = 8/255, step 2/255) and the same evaluation, so differences come from the method only.

---

### 8.1 E0: Standard training (Milestone 1) · ~0.5 h training + ~1 h evaluation
**What happens:** trains the network with plain cross-entropy on clean images, no attacks. The first evaluation also
builds the 30 corrupted test sets (6 corruptions × 5 severities) once and saves them to storage (~1.1 GB).

**Why it is needed**
* It is the **non-robust reference**: how accurate the model is and how easily attacks fool it without any defence.
* Its `best.pt` is the frozen **"OHEM model"** that Fo-Mixup uses to choose which frequency bands to mix (step 8.3).
  So E0 must finish before E4, E5 and the ablations A1–A4.

**Milestone 1 check:** test clean accuracy well above 95 %; PGD-20 expected near 0 %.
Seed-0 result: clean 99.32 %, PGD-20 17.09 %, AutoAttack 10.35 %. PGD-20 is above "near 0"; the analysis (not
gradient masking) is in `NOTES.md` N21.

In [ ]:
# Step 8.1: E0, standard training (no attacks). Baseline + frozen model for the Fo-Mixup masks.
train_eval("E0_standard")

### 8.2 E3: Base adversarial training, the control (Milestone 2) · ~2.5 h training + ~1–1.5 h evaluation
**What happens, per batch**
1. **EAEG attack** (DAT's efficient attack): starting from the image plus tiny noise, take **5 steps** that each
   move every pixel by 2/255 in the direction that increases *cross-entropy + 15 × KL(clean prediction ‖ attacked
   prediction)*, staying within ±8/255 of the original and inside [0, 1]. This gives the adversarial image x′.
2. **Training loss** `L_AT = CE(f(x), y) + 15 · KL(f(x′) ‖ f(x))`: classify the clean image correctly **and** give
   the same prediction on the attacked image.

**Why it is needed:** E3 is the **shared base of both arms**. E5 and E6 use exactly this attack and this loss and
only add their own second branch, so `E5 − E3` and `E6 − E3` measure what each frequency method adds.
It is also DAT's own ablation baseline.

**Milestone 2 check:** robust accuracy clearly above 0 % and all sanity checks pass. Seed-0 result: clean 90.02 %,
PGD-20 66.79 %, AutoAttack 61.55 %, all checks passed, 152 s/epoch.

In [ ]:
# Step 8.2: E3, shared adversarial-training base (EAEG attack + L_AT loss). Control for E5 and E6.
# Milestone 2: E3 (shared adversarial base). Note sec_per_epoch to plan the remaining tiers.
train_eval("E3_base")

### 8.3 Fo-Mixup band masks + visual check (Milestone 3a) · ~5–10 min
**What happens, for every one of the 35,219 training images** (`src/precompute_fomixup.py`)
1. Takes the 2-D Fourier transform of each colour channel and splits the 32×32 spectrum into **144 segments**
   (12 rings by distance from the centre × 12 angular sectors).
2. Groups the 144 segments into **8 clusters** by their average log-amplitude (1-D K-means).
3. For each cluster, keeps only that cluster's frequencies and turns them back into an image (a "band-only" image).
4. Feeds the 8 band-only images to the frozen **E0** model and keeps the **3 clusters with the highest loss**,
   i.e. the bands the standard model can recognise the sign from least.
5. Saves the cluster of every segment and the 3 chosen clusters per image (`data_cache/fomixup_masks.pt`).

During E5 training one of the 3 stored bands is the mask **B**: the image keeps its own amplitude inside B and is
mixed with a partner image *outside* B, which is where the frequencies the model relies on most lie
(FreqDebias Eq. 1, implemented exactly as printed).

**Why it is needed:** this is what makes Fo-Mixup *targeted*: it perturbs the frequency bands the model depends on
most, instead of the whole spectrum (ablations A1 and A2 test whether this matters). It is done once, offline,
because the bands depend only on the image and the frozen E0 model.

**Visual check:** `fomixup_grid.png` shows 16 signs: the original, the same-class partner, the mixed image at
ξ = 0.25 / 0.5 / 0.75 and the mask B (white = kept). **The sign's shape must stay readable** while its colour,
contrast and texture change. To redraw only the grid after a code update:
`py("src.make_grids", "--data", DATA, "--out", OUT, "--which", "fomixup")`.

In [ ]:
# Step 8.3: Fo-Mixup dominant-band masks from the frozen E0 model (once), then the visual check grid.
# Needs E0's best.pt. Needed by E4, E5 and ablations A1-A4.
# Milestone 3a: Fo-Mixup band masks from E0 (needed by E4, E5, A1-A4) + visual check: signs must stay readable.
py("src.precompute_fomixup", "--data", DATA, "--out", OUT, *SMOKE)
show_image(f"{OUT}/figures/fomixup_grid.png")

### 8.4 E5: FreqDebias-AT, Arm A (Milestone 3b) · ~3–5 h training + ~1–1.5 h evaluation
**What happens, per batch**
1. **Fo-Mixup:** for each image x, pick a random training image of the **same class** and build x̃ by mixing the
   two amplitudes outside the mask B with a random weight ξ ~ U(0, 1), plus slight multiplicative noise, while
   keeping x's **phase**. The sign's shape stays, its style changes.
2. **Confidence sampling:** keep the 50 % of the x̃ in the batch that the model is most confident about (lowest
   prediction entropy); the rest are left out of the x̃ terms, so badly distorted samples do not dominate.
3. **Attack both branches** with the same EAEG as E3: x′ from x, and x̃′ from x̃.
4. **Loss** = ½ · (L_AT(x) + L_AT(x̃)) + 1.0 · KL between the softened predictions on x̃ and x (temperature 4)
   + 0.1 · CAM consistency (the class-activation maps of x and x̃ for the true class should agree, measured with
   Jensen–Shannon divergence).
5. **Dual BatchNorm:** clean images use BatchNorm set A, Fo-Mixup images use set B. At test time only **A** is used,
   so the deployed model is an ordinary ResNet-18.

**Why it is needed:** this is **Arm A** of the comparison: style perturbation by *mixing real amplitudes in the
dominant bands*. Compared with E3 it tells whether Fo-Mixup adds robustness to attacks, to corruptions, or both.

**Milestone 3 check:** training is stable (no NaN in the loss columns of `train_log.csv`) and sanity checks pass.

In [ ]:
# Step 8.4: E5, FreqDebias-AT (Arm A): Fo-Mixup branch + consistency losses on top of the E3 base.
train_eval("E5_fd_at")

### 8.5 E6: DAT, Arm B (Milestone 4) · ~4–6 h training (estimate) + ~1–1.5 h evaluation, then the DAT grid
**What happens, per batch** (details ported from DAT's official code)
1. **Attack the clean branch** with EAEG → x′ (BatchNorm A).
2. **Generate an adversarial amplitude:** a small 4-layer generator network takes random noise plus the model's
   prediction on x and outputs a full amplitude spectrum A_G. It is mixed with the image's own amplitude,
   A_mix = λ·A_G + (1 − λ)·A with λ ~ U(0, 1), and turned back into an image x̂ with the **original phase**
   (shape kept, style changed).
3. **Attack the generated branch** with EAEG → x̂′ (BatchNorm B).
4. **Loss** = ½ · (L_AT(x) + L_AT(x̂)) + 2 · Jensen–Shannon divergence between the predictions on x and x̂.
5. **Two players:** the classifier takes a step that **decreases** this loss; the generator takes its own step that
   **increases** it, so it keeps producing the amplitudes the classifier currently handles worst.

After training, the cell draws `dat_grid.png`: 16 signs with x̂ at λ = 0.25 / 0.5 / 0.75 using the trained
generator.

**Why it is needed:** this is **Arm B** of the comparison: style perturbation that is *learned and adversarial*
over the whole spectrum, the opposite design choice to Fo-Mixup's fixed, targeted mixing.

**Milestone 4 check:** stable training, sanity checks pass, and in the grid the sign stays readable while its
colour and contrast change.

In [ ]:
# Step 8.5: E6, DAT (Arm B): adversarial amplitude generator + JS consistency on top of the E3 base,
# then the DAT visual check grid drawn with the trained generator.
# Milestone 4: DAT. The grid is drawn with the trained generator after E6 finishes.
train_eval("E6_dat")
py("src.make_grids", "--data", DATA, "--out", OUT, "--which", "dat", *SMOKE)
show_image(f"{OUT}/figures/dat_grid.png")

### 8.6 Tables and figures (Milestone 5) · ~1 min
**What happens:** `src/aggregate.py` reads every `metrics.json` and writes the thesis tables (Markdown and CSV) and
figures to `results/`, then shows the main table here:
* `table_main`: clean, FGSM, PGD-20, PGD-100, CW∞, AutoAttack, 6-px patch for E0–E6 (best value per column in bold);
* `table_corruption`, `table_safety_classes` (Stop, Yield, No entry, speed limits), `table_cost`, `table_ablation`;
* `fig_eps_curve.png`, `fig_corruption_severity.png`, `fig_train_curves.png`.

With several seeds, every number is shown as mean ± standard deviation and the number of seeds is given.

**Why it is needed:** it turns the individual runs into the comparison that answers the research questions.
Re-run it any time after new runs finish; it always uses everything evaluated so far.

In [ ]:
# Step 8.6: build the thesis tables and figures from every evaluated run, show the main table.
# Milestone 5: tables and figures from everything evaluated so far.
py("src.aggregate", "--out", OUT)
p = f"{OUT}/results/table_main.md"
if os.path.exists(p):
    from IPython.display import Markdown, display
    display(Markdown(open(p, encoding="utf-8").read()))

### 8.7 Environment record (run once)
**What happens:** saves the exact list of installed Python packages (`pip freeze`) to `env/requirements.lock.txt`
and uploads it.

**Why it is needed:** reproducibility for the thesis (`CLAUDE.md` §2.3). Together with the torch / torchvision /
Python versions, GPU name and code commit stored in every `metrics.json`, anyone can rebuild the exact environment.

In [ ]:
# Step 8.7: record the exact package versions for the thesis (run once).
# Environment record for the thesis (run once, after the first successful run).
os.makedirs(f"{OUT}/env", exist_ok=True)
with open(f"{OUT}/env/requirements.lock.txt", "w") as f:
    f.write(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
py("src.storage", "push", "--out", OUT)

## Tier 2: standard baselines and H2 (E1, E2, E4; seed 0) · ~14–16 h in total (estimate)
**What happens:** trains and evaluates three more methods, one after another:
* **E1 PGD-AT** (Madry et al.): the classic adversarial training: 10-step PGD attack, train on the attacked
  images only.
* **E2 TRADES**: 10-step attack on the KL divergence, loss = CE on clean + 6 · KL(attacked ‖ clean).
* **E4 FreqDebias-Clean**: E5 **without any attacks** (needs the Fo-Mixup masks from step 8.3).

**Why it is needed:** E1 and E2 are the standard published defences every robustness paper compares with
(hypothesis H1: do the frequency arms beat them?). E4 tests **H2**: Fo-Mixup was designed for generalisation, so
on its own it should give little adversarial robustness.

In [ ]:
# Tier 2: E1 PGD-AT, E2 TRADES, E4 FreqDebias without attacks (seed 0).
for exp in ["E1_pgdat", "E2_trades", "E4_fd_clean"]:
    train_eval(exp)

## Tier 3: error bars for the core comparison (seeds 1, 2 of E3, E5, E6) · ~30–35 h (estimate)
**What happens:** repeats E3, E5 and E6 with random seeds 1 and 2 (different initialisation, data order and
attack noise).

**Why it is needed:** with one seed, differences of 1–2 points between methods can be pure chance. Three seeds give
mean ± standard deviation, which shows whether the E5 vs E6 difference is real.

In [ ]:
# Tier 3: seeds 1 and 2 of E3, E5, E6, for mean +/- std on the core comparison.
for exp in ["E3_base", "E5_fd_at", "E6_dat"]:
    for seed in [1, 2]:
        train_eval(exp, seed)

## Tier 4: ablations A1–A6 (seed 0) · ~35–40 h in total (estimate)
**What happens:** each ablation removes or replaces one part of E5 or E6 and is compared with its parent:

| Run | Change | Question |
| --- | --- | --- |
| A1 | E5 mixing the **whole spectrum** (no band mask) | Does targeting bands matter? (H4) |
| A2 | E5 with **random** bands instead of the dominant ones | Does choosing the dominant bands matter? (H4) |
| A3 | E5 **without CAM consistency** | Value of the CAM loss |
| A4 | E5 **without confidence sampling** | Value of keeping only confident samples |
| A5 | E6 with a **single BatchNorm** | DAT's largest ablation effect, on GTSRB |
| A6 | E6 with the generator replaced by a **random same-class image's amplitude** | Value of the learned generator |

**Why it is needed:** it explains *why* the arms differ, not just *which* is better. If time is short, A1 and A2
(hypothesis H4) are the most important.

In [ ]:
# Tier 4: ablations A1-A6 (seed 0). A1-A4 need the Fo-Mixup masks; A1 and A2 answer H4.
for exp in ["A1", "A2", "A3", "A4", "A5", "A6"]:
    train_eval(exp)

## Tier 5: error bars for the baselines (seeds 1, 2 of E1, E2) · ~20 h (estimate)
**What happens / why:** repeats E1 and E2 with seeds 1 and 2, so the baselines in the main table also have
mean ± standard deviation. Lowest priority.

In [ ]:
# Tier 5: seeds 1 and 2 of E1 and E2.
for exp in ["E1_pgdat", "E2_trades"]:
    for seed in [1, 2]:
        train_eval(exp, seed)

## Status and final push (run any time)
**What happens:** prints the status of every run (epochs done, DONE or partial, headline metrics), shows what is
in storage compared with the local disk, and uploads anything not yet uploaded.

**Why it is needed:** a quick progress check, and a safe last step before closing the session so nothing is left
only on the Colab machine. To check a run's progress without stopping it, you can also open
`runs/<EXP>/seed0/train_log.csv` in your Hugging Face dataset in the browser.

In [ ]:
# Status of all runs + upload anything not yet in storage. Safe to run any time.
status()
py("src.storage", "status", "--out", OUT)
py("src.storage", "push", "--out", OUT)

---
# Extension: colour-aware frequency mixing (CLAUDE.md §14)
**Idea.** E5 and E6 take the Fourier transform of each **R, G, B channel separately** and change each channel's
amplitude independently. A sign's colour lives in the *relationship* between R, G and B, so this also shifts its
colour, and for traffic signs colour carries meaning (red = prohibition/warning, blue = mandatory,
yellow = priority, white = end of restriction). The colour-aware variants apply the same amplitude mixing in
**YCbCr** instead: **Y** (brightness/texture) is mixed exactly as E5/E6 mix each channel; **Cb, Cr** (colour) are
left untouched. Then the image is converted back to RGB.

**Plan, in order** (each step has its own cell below):
| Phase | What | GPU? |
| --- | --- | --- |
| 0 | **Diagnosis**: do E5/E6 lose more accuracy on red/blue signs than on white ones, relative to E3? | no |
| 1 | Visual check of RGB vs colour-aware mixing (the code is already in the repository) | no |
| 2 | Train **E5-CA** and **E6-CA** (seed 0), identical to E5/E6 except the colour space | yes, ~5–6 h each on a T4 |
| 3 | Overall and per-colour comparison tables | no |

**Run Phase 0 first and decide before spending GPU time.** If Phase 0 shows no gap, the colour-aware variants have
weak motivation, and it is better to know that now than after ~12 GPU hours.

## Phase 0: colour-group diagnosis (no training, ~1 minute)
**What happens:** `src/colour_groups.py` uses results you already have. For every evaluated run (seed 0) it reads
the per-image test results saved by evaluation (`eval_flags.pt`: was each of the 12,630 test images classified
correctly, clean and under PGD-20, by `best.pt`) and groups the 43 classes by sign colour:

| Group | Classes | Test images |
| --- | --- | --- |
| red | prohibitory and warning signs, Yield, Stop, No entry (30 classes) | most of the test set |
| blue | mandatory signs: turn/ahead/keep/roundabout (33–40) | ~1,800 |
| yellow | Priority road (12) | ~690 |
| white | end-of-restriction signs (6, 32, 41, 42) | 360 |

The map was checked against the official class names (test in `tests/test_colour_aware.py`).

It writes two tables to `results/`:
* `table_colour_groups`: clean and PGD-20 accuracy per colour group, per method;
* `table_colour_drop`: for E5 and E6, the difference from E3 per group with a **95 % paired bootstrap interval**
  (all methods are scored on the same test images, so images are resampled jointly), plus the **colour gap** =
  drop on red+blue minus drop on white.

**How to read it (the decision):**
* **colour gap < 0 with an interval that excludes 0** ("colour-coded signs lose MORE"): the gap is confirmed for
  that method, so go on to Phases 1–3.
* **interval includes 0** ("no clear gap"): no evidence that colour is the problem; stop and reconsider before
  training. Caveats: the white group is small (360 images) and yellow is one class, so their intervals are wide;
  this is one seed.

In [ ]:
# Phase 0: per-colour-group accuracy and drops of E5/E6 vs E3 (no GPU, uses existing evaluation results).
py("src.colour_groups", "--out", OUT)

## Phase 1: visual check of colour-aware mixing (~1 minute)
**What happens:** draws `figures/colour_aware_grid.png`: for one sign per class, the original, its same-class
partner, then **RGB mixing (what E5/E6 do)** next to **colour-aware mixing**, for Fo-Mixup and for a DAT-style mix
(using the partner's amplitude, so no trained generator is needed).

**Check:** in the colour-aware columns the sign must keep its red/blue/yellow colour while texture and brightness
change; in the RGB columns the colour shifts.

**What is in the code (already in the repository):**
* `src/color.py`: differentiable RGB ↔ YCbCr (BT.601 full range), float32 like all Fourier arithmetic.
* `src/fomixup.py::fomixup_color`: Fo-Mixup (Eq. 1–2) applied per YCbCr channel; the band masks B are the same as
  for E5.
* `src/methods/dat.py`: with `color_space: ycbcr` the DAT generator outputs **one** channel (the Y amplitude) and
  only Y is mixed.
* The knob **α = `chroma_protect`** in the config: Y is always mixed fully; Cb/Cr are mixed with strength (1 − α).
  α = 1 (used for E5-CA/E6-CA) leaves colour untouched; α = 0 mixes all three YCbCr channels. The original
  per-RGB-channel method is E5/E6 itself (`NOTES.md` N23).

In [ ]:
# Phase 1: RGB vs colour-aware mixing, side by side (needs the Fo-Mixup masks from step 8.3).
py("src.make_grids", "--data", DATA, "--out", OUT, "--which", "ca", *SMOKE)
show_image(f"{OUT}/figures/colour_aware_grid.png")

## Phase 2: train E5-CA and E6-CA (seed 0) · ~5–6 h each on a T4 (estimate; an L4 is ~2× faster)
**What happens:** the usual `train_eval` for the two colour-aware variants. Their configs
(`configs/E5_fd_at_ca.yaml`, `configs/E6_dat_ca.yaml`) are **copies of E5 and E6 with only two added keys**
(`color_space: ycbcr`, `chroma_protect: 1.0`): same adversarial base (EAEG + L_AT), same 60-epoch schedule, same
dual BatchNorm, same masks, same evaluation. That is what makes the comparison fair.

**Why it is needed:** E3, E5 and E6 are already trained and serve as the baselines; these two runs are the only new
GPU cost. Run one cell at a time; each resumes after a disconnect like every other run.

In [ ]:
# Phase 2a: colour-aware FreqDebias-AT (E5 with the mix applied to Y only).
train_eval("E5_fd_at_ca")

In [ ]:
# Phase 2b: colour-aware DAT (E6 with a Y-only generator and mix).
train_eval("E6_dat_ca")

## Phase 3: comparison tables (~1 minute)
**What happens:** rebuilds all tables. Two are specific to this extension:
1. `table_colour_aware`: overall clean, PGD-20, AutoAttack and corruption mean of E5-CA / E6-CA and the difference
   from their parents E5 / E6: **did colour-aware mixing keep the robustness?**
2. `table_colour_drop` (rows `E5_fd_at_ca vs E5_fd_at`, `E6_dat_ca vs E6_dat`): the per-colour-group difference
   with paired bootstrap intervals: **did colour-aware mixing recover the colour-coded classes?**

**Success condition:** the CA variant recovers accuracy on the colour-coded groups (positive Δ on red/blue whose
interval excludes 0) **without** losing overall robustness (AutoAttack and PGD-20 within noise of the parent).
With one seed, treat differences under ~1 point as noise; seeds 1 and 2 would be needed for a firm claim.

In [ ]:
# Phase 3: all tables, then the two colour-aware comparisons.
py("src.aggregate", "--out", OUT)
from IPython.display import Markdown, display
for t in ["table_colour_aware", "table_colour_groups", "table_colour_drop"]:
    p = f"{OUT}/results/{t}.md"
    if os.path.exists(p):
        display(Markdown(f"**{t}**\n\n" + open(p, encoding="utf-8").read()))
    else:
        print(t, ": not available yet (needs the runs above to be evaluated)")